## 1. Matrix Condition Number

The condition number measures how sensitive a matrix is to input perturbations — 
a high value indicates the matrix is nearly singular (ill-conditioned) and small 
errors in input can cause large errors in the solution.

$$
\kappa(A) = \frac{\sigma_{max}(A)}{\sigma_{min}(A)}
$$

where $\sigma_{max}$ and $\sigma_{min}$ are the largest and smallest singular values of $A$.

In [2]:
import numpy as np

A_good = np.array([[1., 0.],
                   [0., 1.]])

A_bad = np.array([[1., 1.],
                  [1., 1.0001]])

print("cond(A_good):", np.linalg.cond(A_good))
print("cond(A_bad): ", np.linalg.cond(A_bad))

cond(A_good): 1.0
cond(A_bad):  40002.000074915224


In [3]:
A_bad = [[1., 1.],
         [1., 1.0001]]


a11 = A_bad[0][0]**2 + A_bad[1][0]**2
a12 = A_bad[0][0]*A_bad[0][1] + A_bad[1][0]*A_bad[1][1]
a22 = A_bad[0][1]**2 + A_bad[1][1]**2

trace = a11 + a22
det   = a11 * a22 - a12 * a12
disc  = (trace**2 - 4*det) ** 0.5

lambda_max = (trace + disc) / 2
lambda_min = (trace - disc) / 2


sigma_max = lambda_max ** 0.5
sigma_min = lambda_min ** 0.5

condition_number = sigma_max / sigma_min
print("Condition number (manual, from A_bad):", condition_number)

Condition number (manual, from A_bad): 40001.99674490561


## 2. Singular Value Decomposition (SVD)

SVD decomposes any matrix into three components that reveal its rank, geometry, 
and directions of maximum variance — foundational for PCA, dimensionality reduction, 
and low-rank approximation.

$$
A = U \Sigma V^T
$$

where $U$ and $V$ are orthogonal matrices (rotation) and $\Sigma$ is a diagonal 
matrix of singular values (scaling).

In [4]:
import numpy as np

A = np.array([[3., 0.],
              [0., 0.5]])

U, S, Vt = np.linalg.svd(A)

print("U:\n", U)
print("Singular values:", S)
print("V^T:\n", Vt)
print("Condition number:", S[0] / S[-1])

U:
 [[1. 0.]
 [0. 1.]]
Singular values: [3.  0.5]
V^T:
 [[1. 0.]
 [0. 1.]]
Condition number: 6.0


In [5]:
a, b = 3.0, 0.5

sigma_1 = max(abs(a), abs(b))
sigma_2 = min(abs(a), abs(b))

print("Singular values:", sigma_1, sigma_2)
print("Condition number:", sigma_1 / sigma_2)

Singular values: 3.0 0.5
Condition number: 6.0


<h1 id="sec3">3. Moore–Penrose Pseudoinverse</h1>

When a matrix $A$ is non-square or singular, the standard inverse $A^{-1}$ does not exist. The **Pseudoinverse** ($A^+$) provides the best alternative.

It is computed directly using the SVD of the matrix:
$$
A^+ = V \Sigma^+ U^T
$$
Where $\Sigma^+$ is formed by taking the reciprocal of all non-zero singular values ($1/\sigma_i$) and leaving the zeros as zero.

**Least Squares Solution:**
To solve an overdetermined system $Ax = b$, the pseudoinverse finds the vector $x$ that minimizes the squared error:
$$
x^* = A^+ b
$$

In [6]:
import numpy as np

A = np.array([[1., 1.],
              [1., 2.],
              [1., 3.]])
b = np.array([2., 2.8, 3.9])

A_plus = np.linalg.pinv(A)
x = A_plus @ b

print("A+:\n", A_plus)
print("x*:", x)
print("prediction:", A @ x)
print("residual norm:", np.linalg.norm(A @ x - b))

A+:
 [[ 1.33333333e+00  3.33333333e-01 -6.66666667e-01]
 [-5.00000000e-01  5.55111512e-17  5.00000000e-01]]
x*: [1.   0.95]
prediction: [1.95 2.9  3.85]
residual norm: 0.12247448713915901


In [7]:
A = [[1., 1.], [1., 2.], [1., 3.]]
b = [2., 2.8, 3.9]

a, c = 3., 6.      
d, e = 6., 14.     

# A^T b
p = 2. + 2.8 + 3.9
q = 1*2. + 2*2.8 + 3*3.9

det = a * e - c * d
x0 = (e * p - c * q) / det
x1 = (-d * p + a * q) / det

print([x0, x1])

[1.0, 0.9499999999999993]


In [8]:
import numpy as np

A = np.array([[3., 1.],
              [1., 3.],
              [2., 2.]])

U, s, Vt = np.linalg.svd(A, full_matrices=False)

k = 1
A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

print("Original:\n", A)
print(f"Rank-{k} approximation:\n", A_k)

Original:
 [[3. 1.]
 [1. 3.]
 [2. 2.]]
Rank-1 approximation:
 [[2. 2.]
 [2. 2.]
 [2. 2.]]


## 3. Moore–Penrose Pseudoinverse

The pseudoinverse generalizes matrix inversion to non-square or singular matrices, 
providing the least-squares solution when an exact solution does not exist.

$$
A^{+} = V \Sigma^{+} U^T
$$

used to solve $Ax = b$ as $x = A^{+}b$, minimizing $\|Ax - b\|_2$.

In [9]:
import numpy as np


A = np.array([[3.0, 1.0], 
              [1.0, 2.0]]) 
x = np.array([2.0, 1.0])

def f(x_val):
    return x_val.T @ A @ x_val


grad_analytical = 2 * (A @ x)


eps = 1e-4
grad_numerical = np.zeros_like(x)
for i in range(len(x)):
    x_plus, x_minus = x.copy(), x.copy()
    x_plus[i] += eps
    x_minus[i] -= eps
    grad_numerical[i] = (f(x_plus) - f(x_minus)) / (2 * eps)


hessian = 2 * A

print("Analytical Gradient:", grad_analytical)
print("Numerical Gradient: ", grad_numerical)
print("Hessian Matrix:\n", hessian)

Analytical Gradient: [14.  8.]
Numerical Gradient:  [14.  8.]
Hessian Matrix:
 [[6. 2.]
 [2. 4.]]


In [10]:

x = [2.0, 1.0]

def f(x_val):
    return 3 * x_val[0]**2 + 2 * x_val[1]**2 + 2 * x_val[0] * x_val[1]


grad_analytical = [6 * x[0] + 2 * x[1], 
                   4 * x[1] + 2 * x[0]]


eps = 1e-4
grad_numerical = [0.0, 0.0]
for i in range(2):
    x_plus, x_minus = list(x), list(x)
    x_plus[i] += eps
    x_minus[i] -= eps
    grad_numerical[i] = (f(x_plus) - f(x_minus)) / (2 * eps)


hessian = [[6.0, 2.0],
           [2.0, 4.0]]

print("Analytical Gradient:", grad_analytical)
print("Numerical Gradient: ", [round(g, 4) for g in grad_numerical])
print("Hessian Matrix:", hessian)

Analytical Gradient: [14.0, 8.0]
Numerical Gradient:  [14.0, 8.0]
Hessian Matrix: [[6.0, 2.0], [2.0, 4.0]]


## 5. Gradient Flow in Deep Learning

Gradient flow describes how gradients propagate backward through a neural network 
during training via the chain rule — poor gradient flow leads to vanishing or 
exploding gradients, a key challenge in training deep networks.

$$
\frac{\partial L}{\partial w_l} = \frac{\partial L}{\partial a_L} \cdot 
\prod_{k=l+1}^{L} \frac{\partial a_k}{\partial a_{k-1}} \cdot \frac{\partial a_l}{\partial w_l}
$$

where $L$ is the loss, $a_l$ is the activation at layer $l$, and the product term 
shows why gradients can shrink or grow exponentially with network depth.

In [11]:
import numpy as np

np.random.seed(42)
num_layers = 10
hidden_size = 4
initial_gradient = np.ones((hidden_size, 1))

print("--- Simulating Vanishing Gradient Flow ---")
grad_vanish = initial_gradient.copy()
for i in range(num_layers):
    W = np.random.randn(hidden_size, hidden_size) * 0.2
    grad_vanish = W.T @ grad_vanish 
    print(f"Layer {num_layers - i} gradient norm: {np.linalg.norm(grad_vanish):.6f}")

print("\n--- Simulating Exploding Gradient Flow ---")
grad_explode = initial_gradient.copy()
for i in range(num_layers):
    W = np.random.randn(hidden_size, hidden_size) * 2.5
    grad_explode = W.T @ grad_explode 
    print(f"Layer {num_layers - i} gradient norm: {np.linalg.norm(grad_explode):.2f}")

--- Simulating Vanishing Gradient Flow ---
Layer 10 gradient norm: 0.430579
Layer 9 gradient norm: 0.237249
Layer 8 gradient norm: 0.105408
Layer 7 gradient norm: 0.038511
Layer 6 gradient norm: 0.019768
Layer 5 gradient norm: 0.003293
Layer 4 gradient norm: 0.000925
Layer 3 gradient norm: 0.000316
Layer 2 gradient norm: 0.000086
Layer 1 gradient norm: 0.000029

--- Simulating Exploding Gradient Flow ---
Layer 10 gradient norm: 4.86
Layer 9 gradient norm: 13.02
Layer 8 gradient norm: 27.07
Layer 7 gradient norm: 184.11
Layer 6 gradient norm: 682.41
Layer 5 gradient norm: 1959.65
Layer 4 gradient norm: 10263.23
Layer 3 gradient norm: 66116.68
Layer 2 gradient norm: 218494.24
Layer 1 gradient norm: 643217.79


In [12]:

num_layers = 10
initial_error_signal = 1.0

print("--- Simulating Vanishing Gradient Flow ---")
local_gradient_small = 0.5 
grad_v = initial_error_signal
for layer in range(num_layers, 0, -1):
    grad_v = grad_v * local_gradient_small
    print(f"Layer {layer} received gradient: {grad_v:.6f}")

print("\n--- Simulating Exploding Gradient Flow ---")
local_gradient_large = 2.0 
grad_e = initial_error_signal
for layer in range(num_layers, 0, -1):
    grad_e = grad_e * local_gradient_large
    print(f"Layer {layer} received gradient: {grad_e:.2f}")

--- Simulating Vanishing Gradient Flow ---
Layer 10 received gradient: 0.500000
Layer 9 received gradient: 0.250000
Layer 8 received gradient: 0.125000
Layer 7 received gradient: 0.062500
Layer 6 received gradient: 0.031250
Layer 5 received gradient: 0.015625
Layer 4 received gradient: 0.007812
Layer 3 received gradient: 0.003906
Layer 2 received gradient: 0.001953
Layer 1 received gradient: 0.000977

--- Simulating Exploding Gradient Flow ---
Layer 10 received gradient: 2.00
Layer 9 received gradient: 4.00
Layer 8 received gradient: 8.00
Layer 7 received gradient: 16.00
Layer 6 received gradient: 32.00
Layer 5 received gradient: 64.00
Layer 4 received gradient: 128.00
Layer 3 received gradient: 256.00
Layer 2 received gradient: 512.00
Layer 1 received gradient: 1024.00
